In [2]:
import optuna
import os
import yaml
from box import Box
from audiomancy.constants import CONFIGS
from audiomancy import train
import warnings
warnings.filterwarnings('ignore')

config_path = os.path.join(CONFIGS,'default.yaml') # o donde tengas tu archivo config

with open(config_path,'r') as f:
    args = Box(yaml.safe_load(f))

In [ ]:
from audiomancy.architecture.model import Audiomancer
import time

class TimeEstimator:
    def __init__(self):
        self.start_time = None
        self.completed_trials = 0

    def __call__(self, study, trial):
        if self.start_time is None:
            self.start_time = time.time()
        
        self.completed_trials += 1
        elapsed_time = time.time() - self.start_time
        avg_time_per_trial = elapsed_time / self.completed_trials
        remaining_trials = study.study_direction.n_trials - self.completed_trials
        estimated_remaining_time = avg_time_per_trial * remaining_trials

        print(f"Completed {self.completed_trials} trials. "
              f"Estimated time remaining: {estimated_remaining_time:.2f} seconds.")

def objective(trial):
    params = {
        "channels": trial.suggest_int("channels", 32, 64),
        "channels_time": trial.suggest_categorical("channels_time", [None, 32, 48]),
        "growth": trial.suggest_int("growth", 1, 4),
        "nfft": trial.suggest_categorical("nfft", [1024, 2048, 4096]),
        "wiener_residual": trial.suggest_categorical("wiener_residual", [True, False]),
        "cac": trial.suggest_categorical("cac", [True, False]),
        "depth": trial.suggest_int("depth", 3, 10), 
        "rewrite": trial.suggest_categorical("rewrite", [True, False]),
        "multi_freqs": trial.suggest_categorical("multi_freqs", [[], [1, 2, 4]]),
        "multi_freqs_depth": trial.suggest_int("multi_freqs_depth", 1, 5),
        "freq_emb": trial.suggest_float("freq_emb", 0.1, 0.3),
        "emb_scale": trial.suggest_int("emb_scale", 5, 15),
        "emb_smooth": trial.suggest_categorical("emb_smooth", [True, False]),
        "kernel_size": trial.suggest_int("kernel_size", 5, 10),
        "stride": trial.suggest_int("stride", 2, 5),
        "time_stride": trial.suggest_int("time_stride", 1, 3),
        "context": trial.suggest_int("context", 0, 2),
        "context_enc": trial.suggest_int("context_enc", 0, 2),
        "norm_starts": trial.suggest_int("norm_starts", 3, 5),
        "norm_groups": trial.suggest_int("norm_groups", 1, 8),
        "dconv_mode": trial.suggest_categorical("dconv_mode", [1, 2, 3]),
        "dconv_depth": trial.suggest_int("dconv_depth", 1, 4),
        "dconv_comp": trial.suggest_int("dconv_comp", 4, 16),
        "dconv_init": trial.suggest_loguniform("dconv_init", 1e-4, 1e-2),
        "bottom_channels": trial.suggest_int("bottom_channels", 0, 128),
        "t_layers": trial.suggest_int("t_layers", 3, 7),
        "t_hidden_scale": trial.suggest_float("t_hidden_scale", 2.0, 6.0),
        "t_heads": trial.suggest_int("t_heads", 4, 12),
        "t_dropout": trial.suggest_float("t_dropout", 0.0, 0.3),
        "t_emb": trial.suggest_categorical("t_emb", ["sin", "cape", "scaled"]),
        "t_norm_first": trial.suggest_categorical("t_norm_first", [True, False]),
        "t_norm_out": trial.suggest_categorical("t_norm_out", [True, False]),
        "t_weight_decay": trial.suggest_float("t_weight_decay", 0.0, 1e-2),
        "t_cross_first": trial.suggest_categorical("t_cross_first", [True, False]),
        "rescale": trial.suggest_float("rescale", 0.05, 0.2),
    }

    extra = {
        'sources': list(args.dset.sources),
        'audio_channels': args.dset.channels,
        'samplerate': args.dset.samplerate,
        'segment': args.dset.segment,
    }
    

    solver = train.get_solver(args)
    solver.model = Audiomancer(**extra, **params)
    metrics = solver.train(optuna=True)
    
    nsdrs = []
    for source in solver.model.sources:
        key = f'nsdr_{source}'
        nsdrs.append(metrics[key])
    
    return nsdrs

time_estimator = TimeEstimator()

study = optuna.create_study(study_name='all-stems',storage='sqlite:///stems.db',directions=['maximize','maximize','maximize','maximize','maximize','maximize'])
study.optimize(objective,n_trials=100,catch=(ValueError,AssertionError),callbacks=[time_estimator],n_jobs=4)

[I 2024-11-18 18:19:46,526] A new study created in RDB with name: all-stems


Creada carpeta temporal para resguardar memoria.

Reconstrucción de audios.


100%|██████████| 50/50 [01:22<00:00,  1.66s/it]
INFO:audiomancy.train:train/valid set size: 35 8
DEBUG:audiomancy.architecture.solver:Checkpoint will be saved to d:\repos\universidad\music-source-separation\outputs\audiomancer0_1\checkpoint.th
